In [2]:
import os
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
from jdcoot.scenario import generate_data

E0000 00:00:1766531098.352056 3218302 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766531098.357484 3218302 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## discrete

In [10]:
data = generate_data()
data_source = data[0]
data_target = data[1]

In [15]:
from jdcoot.utils import xcolumns, discrete_classifier, discrete_accuracy
import numpy as np
from jdcoot.utils import xcolumns, discrete_classifier, discrete_accuracy
from jdcoot.coot import init_matrix_np
from jdcoot.losses import loss_crossentropy2
from tf_keras.utils import to_categorical


def one_hot(y, nClass):
    return to_categorical(y, num_classes=nClass)


def one_cold(z_encoded):
    return np.argmax(z_encoded, axis=1)
x_source = data_source.loc[:, xcolumns(data_source)].values

x_target = data_target.loc[:, xcolumns(data_target)].values

max_diff2 = max(
    (x_source.max() - x_target.min())**2,
    (x_source.min() - x_target.max())**2
)
max_diff2

z1 = np.arange(2)
z2 = np.arange(2)
n_class = 2
Z1 = one_hot(z1, n_class)  # (10, 10)
Z2 = one_hot(z2, n_class)
fcost = np.max(loss_crossentropy2(Z1, Z2))

alpha = fcost/max_diff2
alpha

np.float64(0.43691259978891284)

In [ ]:
alpha_values = np.linspace(0, 1, 11)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique

for a in alpha_values:

    pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(*data, alpha=a)

    score = pure_target  
    
    if score > best_score:
        best_score = score
        best_alpha = a

##continous


In [37]:
from jdcoot.utils import xcolumns, continuous_accuracy, continuous_classifiers
import ot
y_source = data_source.Y.values[:, np.newaxis]
y_target = data_target.Y.values[:, np.newaxis]
max(y_target)-min(y_source)
(max(y_source)-min(y_target))**2
fcost = np.max(ot.dist(y_source, y_target, metric="sqeuclidean"))
fcost
alpha = fcost/max_diff2
alpha

np.float64(513.4604766338035)

In [ ]:
alpha_values = np.linspace(0, 1, 11)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique

for a in alpha_values:

    pure_source, pure_target, test_source, test_target = continuous_unsupervised_jdcoot(*data, alpha=a)

    score = pure_target  
    
    if score > best_score:
        best_score = score
        best_alpha = a